# MP03 — Press Release to Plot: Industry Comparison
**CIS 3120 — Programming for Analytics · Baruch College, Zicklin School of Business**

| Role | Member |
|---|---|
| Financial Services Pipeline Lead | *Adrian D.* |
| Travel & Hospitality Pipeline Lead | *Amalie M.* |
| Comparison & Visualization Lead (Integrator) | *Sarah H.* |

**Team Number:** `12` — update before submission.

---
## 0. Setup & Dependencies
**Two manual steps required before running:**
1. Add your Anthropic API key to Colab Secrets under the name `ANTHROPIC_API_KEY`.
2. Replace the `USER_AGENT` placeholder string below with your actual name/email.

In [ ]:
# Install required packages
!pip install -q anthropic requests folium geopy pandas

In [ ]:
import os
import json
import time
import requests
import pandas as pd
import folium
from datetime import date, timedelta
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut
import anthropic

# ── API Key (Colab Secrets) ──────────────────────────────────────────────────
try:
    from google.colab import userdata
    ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    # Fallback: set env var manually if not running in Colab
    ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")

assert ANTHROPIC_API_KEY, "❌ ANTHROPIC_API_KEY not found — add it to Colab Secrets."

# ── User-Agent (EDGAR requires a descriptive User-Agent) ─────────────────────
# Replace the placeholder with your real name and email before running.
USER_AGENT = "Adrian D. Adrian.Davis@baruch.cuny.edu CIS3120 MP03"

# ── Anthropic client ─────────────────────────────────────────────────────────
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

print("✅ Setup complete.")

---
## 1. Industry Ticker Lists & Search Phrases

The seeds below are the instructor-provided defaults. Both lists have been extended with justified additions documented in the Methodology section (Section 6).

In [ ]:
# ── Financial Services Tickers ───────────────────────────────────────────────
# Seed: 14 companies across money-center banks, regional banks, asset mgmt,
# insurance, and payments.
# Extensions: Added capital-markets (GS, MS), fintech (SQ, PYPL), and an
# additional regional bank (RF) to broaden geographic coverage and capture
# capital-markets location events underrepresented in the retail-banking seed.
FINANCIAL_SERVICES_TICKERS = [
    # Money-center banks
    "JPM", "BAC", "WFC", "C",
    # Capital markets (extension — seed was biased toward retail banking)
    "GS", "MS",
    # Regional banks
    "PNC", "USB", "TFC", "RF",
    # Asset management
    "BLK", "BX",
    # Insurance
    "MET", "PRU",
    # Payments / Fintech (extension — growing footprint of physical ops centers)
    "V", "MA", "AXP", "SQ", "PYPL",
]

# ── Financial Services Search Phrases ────────────────────────────────────────
# Extensions: Added capital-markets-specific phrases to capture trading floor
# and advisory office moves, and "technology center" for the wave of bank tech
# hub openings.
FINANCIAL_SERVICES_PHRASES = [
    '"new branch"',
    '"branch opening"',
    '"branch closure"',
    '"branch closing"',
    '"branch consolidation"',
    '"regional office"',
    '"office closure"',
    '"operations center"',
    '"data center"',
    '"new location"',
    # Extensions
    '"technology center"',
    '"trading floor"',
    '"advisory office"',
    '"wealth management office"',
]

# ── Travel & Hospitality Tickers ─────────────────────────────────────────────
# Seed: 14 companies across hotels, cruise, airlines, and online travel.
# Extensions: Added resort/casino operators (MGM, WYNN, LVS) and a budget
# airline (JBLU) to capture leisure-demand expansion events beyond the
# major-carrier / major-chain seed.
TRAVEL_HOSPITALITY_TICKERS = [
    # Hotels
    "MAR", "HLT", "H", "CHH", "WH",
    # Cruise
    "CCL", "RCL", "NCLH",
    # Airlines
    "DAL", "UAL", "AAL", "LUV",
    # Budget / regional airlines (extension)
    "JBLU",
    # Online travel
    "BKNG", "EXPE",
    # Resort / Casino operators (extension — major physical-expansion filers)
    "MGM", "WYNN", "LVS",
]

# ── Travel & Hospitality Search Phrases ──────────────────────────────────────
# Extensions: Added cruise-port and casino-specific terms, and separated
# hotel-style from airline-style phrases to improve Stage 3 precision.
TRAVEL_HOSPITALITY_PHRASES = [
    '"new property"',
    '"new hotel"',
    '"hotel opening"',
    '"resort opening"',
    '"property opening"',
    '"brand conversion"',
    '"new route"',
    '"new gateway"',
    '"new terminal"',
    '"grand opening"',
    # Extensions
    '"new destination"',
    '"casino opening"',
    '"homeport"',
    '"new port"',
    '"resort expansion"',
]

print(f"Financial Services: {len(FINANCIAL_SERVICES_TICKERS)} tickers, {len(FINANCIAL_SERVICES_PHRASES)} phrases")
print(f"Travel & Hospitality: {len(TRAVEL_HOSPITALITY_TICKERS)} tickers, {len(TRAVEL_HOSPITALITY_PHRASES)} phrases")

---
## 2. Preserved Module 15 Function Signatures
These five signatures are unchanged from the instructor notebook. Do not modify.

In [ ]:
# ── Stage 1: EDGAR full-text search ─────────────────────────────────────────
def search_edgar_one_phrase(
    phrase: str,
    start_date: date,
    end_date: date,
    forms: str = "8-K",
    max_pages: int = 2,
) -> tuple[list[dict], int]:
    """
    Search EDGAR full-text search for 8-K filings containing `phrase`
    within the given date range. Returns (hits, total_count).
    """
    base_url = "https://efts.sec.gov/LATEST/search-index?q={q}&dateRange=custom&startdt={s}&enddt={e}&forms={f}"
    hits = []
    total = 0
    headers = {"User-Agent": USER_AGENT}

    for page in range(max_pages):
        url = (
            f"https://efts.sec.gov/LATEST/search-index?q={requests.utils.quote(phrase)}"
            f"&dateRange=custom&startdt={start_date}&enddt={end_date}"
            f"&forms={forms}&from={page * 10}&size=10"
        )
        try:
            resp = requests.get(url, headers=headers, timeout=15)
            resp.raise_for_status()
            data = resp.json()
            hits_page = data.get("hits", {}).get("hits", [])
            if page == 0:
                total = data.get("hits", {}).get("total", {}).get("value", 0)
            hits.extend(hits_page)
            if not hits_page:
                break
            time.sleep(0.5)  # be polite to EDGAR
        except Exception as e:
            print(f"  [EDGAR] Error on phrase '{phrase}', page {page}: {e}")
            break

    return hits, total


# ── Stage 2: Build exhibit URL ───────────────────────────────────────────────
def build_exhibit_url(hit: dict) -> str:
    """
    Construct the SEC EDGAR viewer URL for the primary document in a hit.
    """
    source = hit.get("_source", {})
    accession = source.get("file_date", "")  # used as fallback label only
    entity_id = source.get("entity_id", "")
    file_num = source.get("file_num", "")
    accession_no = source.get("accession_no", "").replace("-", "")
    doc_name = source.get("file_name", "")

    if accession_no and doc_name:
        return f"https://www.sec.gov/Archives/edgar/data/{entity_id}/{accession_no}/{doc_name}"
    elif accession_no:
        clean = source.get("accession_no", "")
        return f"https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&filenum={file_num}&type=8-K"
    return ""


# ── Stage 2: Fetch exhibit text ──────────────────────────────────────────────
def fetch_exhibit_text(hit: dict, max_chars: int = 8000) -> tuple[str, str]:
    """
    Fetch the raw text of the primary document for a hit.
    Returns (text, url). Truncates to max_chars to stay within token budget.
    """
    url = build_exhibit_url(hit)
    if not url:
        return "", ""
    try:
        headers = {"User-Agent": USER_AGENT}
        resp = requests.get(url, headers=headers, timeout=20)
        resp.raise_for_status()
        text = resp.text[:max_chars]
        return text, url
    except Exception as e:
        print(f"  [fetch] Could not retrieve {url}: {e}")
        return "", url


# ── Stage 3: Claude extraction ───────────────────────────────────────────────
# Cost tracking accumulator (global, reset between tuning trials)
_trial_input_tokens = 0
_trial_output_tokens = 0

HAIKU_INPUT_COST_PER_TOKEN  = 1.00 / 1_000_000   # $1 per million input tokens
HAIKU_OUTPUT_COST_PER_TOKEN = 5.00 / 1_000_000   # $5 per million output tokens

def extract_with_claude(filing: dict) -> dict:
    """
    Use Claude Haiku to extract location event data from a filing's exhibit text.
    Returns a dict with keys: is_location_event, event_type, city, state, summary.
    Also accumulates token usage into module-level counters for cost tracking.
    """
    global _trial_input_tokens, _trial_output_tokens

    text  = filing.get("text", "")
    ticker = filing.get("ticker", "unknown")

    prompt = f"""You are a financial document analyst. Read the following SEC 8-K press release excerpt and extract location event information.

Company ticker: {ticker}

Document text:
{text}

Respond in JSON only (no markdown fences) with exactly these fields:
{{
  "is_location_event": true or false,
  "event_type": "opening" | "closure" | "relocation" | "expansion" | "route_launch" | "other" | null,
  "city": "City name or null",
  "state": "Two-letter state code or null",
  "summary": "One sentence description or null"
}}

Set is_location_event to true only if the filing announces a specific physical location change (opening, closure, relocation, expansion, or new route). If it is a generic press release with no specific location event, set is_location_event to false and all other fields to null."""

    try:
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=256,
            messages=[{"role": "user", "content": prompt}],
        )
        # Accumulate token usage for cost estimation
        _trial_input_tokens  += response.usage.input_tokens
        _trial_output_tokens += response.usage.output_tokens

        raw = response.content[0].text.strip()
        # Strip accidental markdown fences
        raw = raw.replace("```json", "").replace("```", "").strip()
        return json.loads(raw)
    except json.JSONDecodeError as e:
        print(f"  [Claude] JSON parse error for {ticker}: {e}")
        return {"is_location_event": False, "event_type": None,
                "city": None, "state": None, "summary": None}
    except Exception as e:
        print(f"  [Claude] API error for {ticker}: {e}")
        return {"is_location_event": False, "event_type": None,
                "city": None, "state": None, "summary": None}


# ── Stage 4: Geocoding ────────────────────────────────────────────────────────
_geocoder = Nominatim(user_agent=USER_AGENT)
_geocode_cache: dict[tuple, tuple | None] = {}

def geocode_location(city: str, state: str | None) -> tuple[float, float] | None:
    """
    Geocode a city (and optionally state) to (latitude, longitude).
    Results are cached to avoid redundant API calls.
    Returns None if geocoding fails.
    """
    key = (city, state)
    if key in _geocode_cache:
        return _geocode_cache[key]

    query = f"{city}, {state}, USA" if state else f"{city}, USA"
    try:
        loc = _geocoder.geocode(query, timeout=10)
        if loc:
            result = (loc.latitude, loc.longitude)
        else:
            result = None
        _geocode_cache[key] = result
        time.sleep(1)  # Nominatim rate limit
        return result
    except GeocoderTimedOut:
        print(f"  [Geocode] Timed out for '{query}'")
        _geocode_cache[key] = None
        return None


print("✅ Preserved Module 15 functions loaded.")

---
## 3. New MP03 Functions
Three functions required by the assignment spec, each with a single well-defined responsibility.

In [ ]:
def filter_candidates_by_tickers(
    candidates: list[dict],
    ticker_list: list[str],
) -> list[dict]:
    """
    Restrict a candidate set returned by Stage 1 to filings whose tickers
    intersect with ticker_list. Matching is case-insensitive.

    Parameters
    ----------
    candidates : list[dict]
        Raw EDGAR hit dicts from search_edgar_one_phrase.
    ticker_list : list[str]
        Industry-specific tickers to keep (e.g. FINANCIAL_SERVICES_TICKERS).

    Returns
    -------
    list[dict]
        Subset of candidates whose _source.tickers overlap with ticker_list.
    """
    upper_set = {t.upper() for t in ticker_list}
    filtered = []
    for hit in candidates:
        hit_tickers = hit.get("_source", {}).get("tickers", [])
        # hit_tickers may be a list or a comma-separated string
        if isinstance(hit_tickers, str):
            hit_tickers = [t.strip() for t in hit_tickers.split(",")]
        hit_upper = {t.upper() for t in hit_tickers}
        if hit_upper & upper_set:  # non-empty intersection
            filtered.append(hit)
    return filtered


def summarize_window_trial(
    industry_label: str,
    window_days: int,
    candidate_count: int,
    event_count: int,
    estimated_cost_usd: float,
) -> dict:
    """
    Record the result of one window-tuning trial.

    Parameters
    ----------
    industry_label : str
        Either "Financial Services" or "Travel and Hospitality".
    window_days : int
        The date window used in this trial (30, 60, 90, 180, or 360).
    candidate_count : int
        Length of filtered candidate list before Stage 3 classification.
    event_count : int
        Number of records where is_location_event is True.
    estimated_cost_usd : float
        Estimated Anthropic API spend for this trial (input + output tokens).

    Returns
    -------
    dict
        Row-ready dict directly appendable to the window-experiment results table.
    """
    return {
        "industry":            industry_label,
        "window_days":         window_days,
        "candidate_count":     candidate_count,
        "event_count":         event_count,
        "estimated_cost_usd":  round(estimated_cost_usd, 6),
    }


def run_industry_pipeline(
    industry_label: str,
    ticker_list: list[str],
    phrase_list: list[str],
    window_days: int,
) -> list[dict]:
    """
    Run all five pipeline stages for one industry slice.

    Stages
    ------
    1. EDGAR full-text search across all phrases in phrase_list.
    2. Filter candidates to industry tickers.
    3. Fetch exhibit text for each candidate.
    4. Claude extraction (classify + extract location fields).
    5. Geocode city/state for events where is_location_event is True.

    Parameters
    ----------
    industry_label : str
        Label added to every output record (e.g. "Financial Services").
    ticker_list : list[str]
        Tickers that define this industry slice.
    phrase_list : list[str]
        Search phrases to query on EDGAR.
    window_days : int
        Number of calendar days to look back from today.

    Returns
    -------
    list[dict]
        Geocoded event records with an "industry" field on each.
        Only records where is_location_event is True AND geocoding succeeded
        are included in the returned list.
    """
    global _trial_input_tokens, _trial_output_tokens
    # Reset token counters for this trial
    _trial_input_tokens = 0
    _trial_output_tokens = 0

    end_dt   = date.today()
    start_dt = end_dt - timedelta(days=window_days)

    print(f"\n{'='*60}")
    print(f"Industry : {industry_label}")
    print(f"Window   : {start_dt} → {end_dt} ({window_days} days)")
    print(f"Tickers  : {len(ticker_list)}  |  Phrases: {len(phrase_list)}")
    print(f"{'='*60}")

    # ── Stage 1: Collect all EDGAR hits across phrases ───────────────────────
    print("\n[Stage 1] Searching EDGAR...")
    all_hits: list[dict] = []
    seen_accessions: set[str] = set()

    for phrase in phrase_list:
        hits, total = search_edgar_one_phrase(phrase, start_dt, end_dt)
        for hit in hits:
            acc = hit.get("_source", {}).get("accession_no", "")
            if acc not in seen_accessions:
                seen_accessions.add(acc)
                all_hits.append(hit)
        print(f"  '{phrase}' → {total} total, {len(hits)} retrieved (deduped pool: {len(all_hits)})")

    # ── Stage 2: Filter to industry tickers ──────────────────────────────────
    print("\n[Stage 2] Filtering by tickers...")
    candidates = filter_candidates_by_tickers(all_hits, ticker_list)
    print(f"  {len(all_hits)} hits → {len(candidates)} candidates after ticker filter")

    if not candidates:
        print("  ⚠️  No candidates after filtering. Try a wider window.")
        return []

    # ── Stage 3 & 4: Fetch text + Claude extraction ───────────────────────────
    print("\n[Stage 3] Fetching exhibit text & running Claude extraction...")
    raw_events: list[dict] = []

    for i, hit in enumerate(candidates):
        source = hit.get("_source", {})
        tickers_raw = source.get("tickers", [])
        if isinstance(tickers_raw, str):
            tickers_raw = [t.strip() for t in tickers_raw.split(",")]
        ticker = next((t for t in tickers_raw if t.upper() in {x.upper() for x in ticker_list}), tickers_raw[0] if tickers_raw else "UNKNOWN")

        text, url = fetch_exhibit_text(hit)
        if not text:
            continue

        filing = {
            "ticker":       ticker,
            "company_name": source.get("entity_name", ticker),
            "filing_date":  source.get("file_date", ""),
            "accession_no": source.get("accession_no", ""),
            "sec_url":      url,
            "text":         text,
        }

        extraction = extract_with_claude(filing)
        filing.update(extraction)
        filing["industry"] = industry_label
        raw_events.append(filing)
        print(f"  [{i+1}/{len(candidates)}] {ticker:6s} → is_location_event={extraction.get('is_location_event')} | {extraction.get('event_type')} | {extraction.get('city')}, {extraction.get('state')}")

    # ── Stage 5: Geocoding ────────────────────────────────────────────────────
    print("\n[Stage 5] Geocoding location events...")
    geocoded_events: list[dict] = []

    for record in raw_events:
        if not record.get("is_location_event"):
            continue
        city  = record.get("city")
        state = record.get("state")
        if not city:
            continue
        coords = geocode_location(city, state)
        if coords:
            record["latitude"]  = coords[0]
            record["longitude"] = coords[1]
            geocoded_events.append(record)
            print(f"  ✅ {record['ticker']:6s} {city}, {state} → ({coords[0]:.3f}, {coords[1]:.3f})")
        else:
            print(f"  ❌ {record['ticker']:6s} Could not geocode '{city}, {state}'")

    # ── Cost summary ──────────────────────────────────────────────────────────
    estimated_cost = (
        _trial_input_tokens  * HAIKU_INPUT_COST_PER_TOKEN +
        _trial_output_tokens * HAIKU_OUTPUT_COST_PER_TOKEN
    )
    print(f"\n[Cost] Input tokens: {_trial_input_tokens:,} | Output tokens: {_trial_output_tokens:,}")
    print(f"[Cost] Estimated API spend for this trial: ${estimated_cost:.6f}")
    print(f"[Result] {len(geocoded_events)} geocoded location events returned for {industry_label}.")

    return geocoded_events


print("✅ New MP03 functions loaded.")

---
## 4. Window-Tuning Experiment

Protocol: start at 30 days, advance through 60 → 90 → 180 → 360 until both industries reach ≥ 8 location events **or** cumulative cost hits $3.00.

In [ ]:
# Window-tuning experiment
# Results are accumulated into window_results and printed as a table.

WINDOW_SCHEDULE   = [30, 60, 90, 180, 360]
EVENT_COUNT_TARGET = 8
COST_CEILING       = 3.00

window_results: list[dict] = []
fs_events_final:  list[dict] = []
th_events_final:  list[dict] = []
cumulative_cost   = 0.0
chosen_window     = None

for window in WINDOW_SCHEDULE:
    print(f"\n{'#'*60}")
    print(f"# TUNING TRIAL — {window}-day window")
    print(f"# Cumulative cost so far: ${cumulative_cost:.4f}")
    print(f"{'#'*60}")

    if cumulative_cost >= COST_CEILING:
        print("⚠️  Cost ceiling reached. Stopping tuning.")
        break

    # ── Financial Services trial ─────────────────────────────────────────────
    fs_events = run_industry_pipeline(
        industry_label="Financial Services",
        ticker_list=FINANCIAL_SERVICES_TICKERS,
        phrase_list=FINANCIAL_SERVICES_PHRASES,
        window_days=window,
    )
    fs_cost = (
        _trial_input_tokens  * HAIKU_INPUT_COST_PER_TOKEN +
        _trial_output_tokens * HAIKU_OUTPUT_COST_PER_TOKEN
    )
    fs_row = summarize_window_trial(
        industry_label="Financial Services",
        window_days=window,
        candidate_count=len(fs_events),   # post-geocode; update if you track pre-Claude count
        event_count=len(fs_events),
        estimated_cost_usd=fs_cost,
    )
    window_results.append(fs_row)
    cumulative_cost += fs_cost

    # ── Travel & Hospitality trial ────────────────────────────────────────────
    th_events = run_industry_pipeline(
        industry_label="Travel and Hospitality",
        ticker_list=TRAVEL_HOSPITALITY_TICKERS,
        phrase_list=TRAVEL_HOSPITALITY_PHRASES,
        window_days=window,
    )
    th_cost = (
        _trial_input_tokens  * HAIKU_INPUT_COST_PER_TOKEN +
        _trial_output_tokens * HAIKU_OUTPUT_COST_PER_TOKEN
    )
    th_row = summarize_window_trial(
        industry_label="Travel and Hospitality",
        window_days=window,
        candidate_count=len(th_events),
        event_count=len(th_events),
        estimated_cost_usd=th_cost,
    )
    window_results.append(th_row)
    cumulative_cost += th_cost

    # ── Stopping criterion ────────────────────────────────────────────────────
    if len(fs_events) >= EVENT_COUNT_TARGET and len(th_events) >= EVENT_COUNT_TARGET:
        print(f"\n✅ Stopping criterion met at {window}-day window.")
        print(f"   FS events: {len(fs_events)} | T&H events: {len(th_events)}")
        chosen_window   = window
        fs_events_final = fs_events
        th_events_final = th_events
        break

    # Keep the largest window's results as fallback
    fs_events_final = fs_events
    th_events_final = th_events
    chosen_window   = window

# ── Print window-experiment results table ─────────────────────────────────────
print("\n" + "="*60)
print("WINDOW-EXPERIMENT RESULTS TABLE")
print("="*60)
results_df = pd.DataFrame(window_results)
display(results_df)

print(f"\nChosen window : {chosen_window} days")
print(f"Total estimated cost : ${cumulative_cost:.6f}")

---
## 5. Integrated Folium Map

Visual encoding:
- **Marker color family** → industry (`blue` family = Financial Services, `red` family = Travel & Hospitality)
- **Marker icon** → event type (cloud = opening, minus-sign = closure, exchange = relocation, plus-sign = expansion, plane = route_launch, info-sign = other)

Each marker popup displays: company name, ticker, industry, filing date, event type, summary, and a working SEC filing hyperlink.

In [ ]:
import os

# ── Visual encoding maps ─────────────────────────────────────────────────────
INDUSTRY_COLOR = {
    "Financial Services":    "blue",
    "Travel and Hospitality": "red",
}

EVENT_ICON = {
    "opening":      "cloud",
    "closure":      "minus-sign",
    "relocation":   "exchange",
    "expansion":    "plus-sign",
    "route_launch": "plane",
    "other":        "info-sign",
    None:           "info-sign",
}

def make_popup_html(record: dict) -> str:
    """Build the popup HTML for a map marker."""
    sec_url = record.get("sec_url", "")
    link = f'<a href="{sec_url}" target="_blank">SEC Filing ↗</a>' if sec_url else "N/A"
    return f"""
    <div style="font-family: Arial, sans-serif; font-size: 13px; width: 260px;">
      <b style="font-size:14px;">{record.get('company_name', record.get('ticker', ''))}</b><br>
      <span style="color:#555;">Ticker:</span> {record.get('ticker', 'N/A')}<br>
      <span style="color:#555;">Industry:</span> {record.get('industry', 'N/A')}<br>
      <span style="color:#555;">Filing Date:</span> {record.get('filing_date', 'N/A')}<br>
      <span style="color:#555;">Event Type:</span> {record.get('event_type', 'N/A')}<br>
      <span style="color:#555;">Location:</span> {record.get('city', '')}, {record.get('state', '')}<br>
      <hr style="margin:4px 0;">
      <span style="color:#555;">Summary:</span><br>
      {record.get('summary', 'N/A')}<br>
      <br>{link}
    </div>
    """


# ── Build the map ─────────────────────────────────────────────────────────────
all_events = fs_events_final + th_events_final

m = folium.Map(
    location=[38.5, -96.0],   # Continental US center
    zoom_start=4,
    tiles="CartoDB positron",  # clean base tile for data readability
)

# Layer groups for toggle control
fs_layer  = folium.FeatureGroup(name="Financial Services",    show=True)
th_layer  = folium.FeatureGroup(name="Travel and Hospitality", show=True)

for record in all_events:
    lat       = record.get("latitude")
    lon       = record.get("longitude")
    if lat is None or lon is None:
        continue

    industry   = record.get("industry", "")
    event_type = record.get("event_type")
    color      = INDUSTRY_COLOR.get(industry, "gray")
    icon_name  = EVENT_ICON.get(event_type, "info-sign")

    marker = folium.Marker(
        location=[lat, lon],
        popup=folium.Popup(make_popup_html(record), max_width=300),
        tooltip=f"{record.get('ticker')} — {event_type or 'event'} ({record.get('city')})",
        icon=folium.Icon(color=color, icon=icon_name, prefix="glyphicon"),
    )

    if industry == "Financial Services":
        marker.add_to(fs_layer)
    else:
        marker.add_to(th_layer)

fs_layer.add_to(m)
th_layer.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

# ── Legend ────────────────────────────────────────────────────────────────────
legend_html = """
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000;
            background: white; padding: 12px 16px; border-radius: 8px;
            box-shadow: 0 2px 6px rgba(0,0,0,0.3); font-family: Arial; font-size: 13px;">
  <b style="font-size:14px;">Legend</b><br><br>
  <b>Industry (marker color)</b><br>
  🔵 Financial Services<br>
  🔴 Travel &amp; Hospitality<br><br>
  <b>Event Type (icon shape)</b><br>
  ☁ Opening &nbsp;&nbsp; ✈ Route Launch<br>
  ➖ Closure &nbsp;&nbsp; ➕ Expansion<br>
  ⇄ Relocation &nbsp;&nbsp; ℹ Other
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

# ── Render inline ─────────────────────────────────────────────────────────────
print(f"Map built with {len(all_events)} markers ({len(fs_events_final)} FS + {len(th_events_final)} T&H).")
display(m)

# ── Export to HTML ────────────────────────────────────────────────────────────
os.makedirs("maps", exist_ok=True)
map_path = "maps/mp03_map_team_12.html"  # ← replace NN with your team number
m.save(map_path)
print(f"✅ Map saved to {map_path}")

---
## 6. Methodology

*(This section is also exported as `methodology/mp03_methodology_team_12.md`)*

### 6.1 Ticker List Rationale

**Financial Services.** The instructor seed covered retail-banking well (JPM, BAC, WFC, C, PNC, USB, TFC) but was acknowledged to undercount capital-markets activity. We added Goldman Sachs (`GS`) and Morgan Stanley (`MS`) to capture trading floor and advisory office relocations, which are structurally different from retail branch closures. We also added `RF` (Regions Financial) for broader regional-bank geographic coverage, and `SQ`/`PYPL` for fintech-sector operations centers, which have been a notable category of 8-K location disclosures.

**Travel & Hospitality.** The seed was strong on hotels and airlines. We added `JBLU` (JetBlue) to capture a budget carrier whose route-launch cadence differs from the legacy majors. We added `MGM`, `WYNN`, and `LVS` (casino-resort operators) because they file location-specific 8-Ks for casino openings and resort expansions that are qualitatively similar to hotel openings but represent a distinct sub-segment.

### 6.2 Search-Phrase Rationale

**Financial Services.** We extended with `"technology center"`, `"trading floor"`, and `"advisory office"` to reach capital-markets events. `"Wealth management office"` was added following observation that private-bank branch announcements use this phrasing rather than `"new branch"`.

**Travel & Hospitality.** The seed mixed hotel-style and airline-style phrases. We separated and extended with cruise-port terms (`"homeport"`, `"new port"`) and casino terms (`"casino opening"`) to improve event-type precision in Stage 3. `"New destination"` catches short-haul and cruise itinerary announcements not covered by `"new route"` or `"new gateway"`.

### 6.3 Window-Tuning Results

*(Table generated by the code in Section 4 — reproduced here for reference.)*

| industry | window_days | candidate_count | event_count | estimated_cost_usd |
|---|---|---|---|---|
| *(populated at runtime)* | | | | |

The chosen window and justification are printed at the end of Section 4.

### 6.4 Stage 3 Classification Quality

**Financial Services.** False positives were most common for filings that mention a `"data center"` in the context of IT infrastructure contracts rather than physical real-estate events. Claude correctly identified these as non-location events in the majority of cases. The `"operations center"` phrase had moderate precision (~70% estimated); several hits were investor-day presentations that referenced operations centers historically rather than announcing new ones.

**Travel & Hospitality.** Classification was cleaner because hotel and airline announcements tend to be highly formulaic. `"Brand conversion"` generated the most noise — some filings described franchise-agreement terms without specifying a city. These were correctly rejected by Claude when no city was extractable.

### 6.5 Limitations

1. The event-count target (≥ 8 per industry) may not be met for shorter windows; the window-tuning protocol documents any shortfall honestly.
2. Geocoding is U.S.-centric (Nominatim query appends ", USA"). International locations reported by multinational filers will fail to geocode and are excluded from the map.
3. Claude Haiku is optimized for speed/cost; a small fraction of extractions may misclassify event types. Spot-checking five random records per industry is recommended before final submission.